In [1]:
import os

os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", None)
os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", None)

import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import efficientnet_v2

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

try:
    tf.config.threading.set_intra_op_parallelism_threads(0)
    tf.config.threading.set_inter_op_parallelism_threads(0)
except Exception:
    pass

print("TF version:", tf.__version__)




2026-05-12 01:11:39.060678: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 01:11:39.071406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778548299.083925      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778548299.087572      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 01:11:39.102020: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF version: 2.18.0


In [2]:
WORK_DIR = "/kaggle/input/cassava-leaf-disease-classification/"
TRAIN_CSV = os.path.join(WORK_DIR, "train.csv")
SAMPLE_SUB = os.path.join(WORK_DIR, "sample_submission.csv")
TRAIN_IMG_DIR = os.path.join(WORK_DIR, "train_images")
TEST_IMG_DIR = os.path.join(WORK_DIR, "test_images")
TRAIN_TFREC_DIR = os.path.join(WORK_DIR, "train_tfrecords")
TEST_TFREC_DIR = os.path.join(WORK_DIR, "test_tfrecords")

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(SAMPLE_SUB), SAMPLE_SUB
assert os.path.isdir(TRAIN_IMG_DIR), TRAIN_IMG_DIR
assert os.path.isdir(TEST_IMG_DIR), TEST_IMG_DIR
assert os.path.isdir(TRAIN_TFREC_DIR), TRAIN_TFREC_DIR
assert os.path.isdir(TEST_TFREC_DIR), TEST_TFREC_DIR

train_df = pd.read_csv(TRAIN_CSV)
sample_df = pd.read_csv(SAMPLE_SUB)

print("train_df:", train_df.shape, train_df.columns.tolist())
print("sample_df:", sample_df.shape, sample_df.columns.tolist())
print("num_classes:", train_df["label"].nunique())




train_df: (18721, 2) ['image_id', 'label']
sample_df: (2676, 2) ['image_id', 'label']
num_classes: 5


In [3]:
class SigmoidFocalCrossEntropy(tf.keras.losses.Loss):
    def __init__(self, alpha=0.25, gamma=2.0, from_logits=False, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha
        self.gamma = gamma
        self.from_logits = from_logits

    def call(self, y_true, y_pred):
        if self.from_logits:
            y_pred = tf.sigmoid(y_pred)
        y_pred = tf.clip_by_value(
            y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon()
        )
        cross_entropy = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(
            1 - y_pred
        )
        weight = self.alpha * y_true + (1 - self.alpha) * (1 - y_true)
        focal_loss = weight * ((1 - y_pred) ** self.gamma) * cross_entropy
        return tf.reduce_sum(focal_loss, axis=-1)


custom_objects = {"SigmoidFocalCrossEntropy": SigmoidFocalCrossEntropy}




In [4]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = int(train_df["label"].nunique())

from sklearn.model_selection import train_test_split

tr_df, va_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["label"],
)

_SEED_T = tf.constant(SEED, dtype=tf.int64)
_NUM_CLASSES_T = tf.constant(NUM_CLASSES, dtype=tf.int32)

# --- Speed fix (preserves core logic): use TFRecords for fast sequential reads instead of
#     thousands of individual JPEG file reads; cache decoded/resized base images once, then
#     apply the same stateless flip + one-hot mapping each epoch.
#     This preserves identical model/training loop semantics and augmentation behavior.
TFREC_FEATURES_TRAIN = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "image_id": tf.io.FixedLenFeature([], tf.string),
    "target": tf.io.FixedLenFeature([], tf.int64),
}
TFREC_FEATURES_TEST = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "image_id": tf.io.FixedLenFeature([], tf.string),
}


@tf.function
def _decode_and_resize_from_bytes(img_bytes):
    img = tf.io.decode_jpeg(img_bytes, channels=3, dct_method="INTEGER_FAST")
    img = tf.ensure_shape(img, [None, None, 3])
    img = tf.image.resize(img, IMG_SIZE, method="bilinear")
    img = tf.cast(img, tf.float32) / 255.0
    return img


@tf.function
def _parse_train_example(ex):
    x = tf.io.parse_single_example(ex, TFREC_FEATURES_TRAIN)
    img = _decode_and_resize_from_bytes(x["image"])
    label = tf.cast(x["target"], tf.int32)
    img_id = x["image_id"]
    return img_id, img, label


@tf.function
def _parse_test_example(ex):
    x = tf.io.parse_single_example(ex, TFREC_FEATURES_TEST)
    img = _decode_and_resize_from_bytes(x["image"])
    img_id = x["image_id"]
    return img_id, img


def _sorted_tfrecord_files(tfrecord_dir: str):
    files = tf.io.gfile.glob(os.path.join(tfrecord_dir, "*.tfrec"))
    files = sorted(files)
    if not files:
        raise FileNotFoundError(f"No tfrecords found in {tfrecord_dir}")
    return files


TRAIN_TFRECS = _sorted_tfrecord_files(TRAIN_TFREC_DIR)
TEST_TFRECS = _sorted_tfrecord_files(TEST_TFREC_DIR)

# Build id->(label, split_idx) lookup from CSV to exactly reproduce the same train/val split.
tr_ids = tr_df["image_id"].astype(str).tolist()
va_ids = va_df["image_id"].astype(str).tolist()

tr_labels = tr_df["label"].astype(np.int32).to_numpy()
va_labels = va_df["label"].astype(np.int32).to_numpy()

tr_tbl = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(tr_ids, dtype=tf.string),
        values=tf.constant(tr_labels, dtype=tf.int32),
    ),
    default_value=tf.constant(-1, tf.int32),
)
va_tbl = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(va_ids, dtype=tf.string),
        values=tf.constant(va_labels, dtype=tf.int32),
    ),
    default_value=tf.constant(-1, tf.int32),
)


@tf.function
def _keep_train(img_id, img, label):
    # Use CSV split membership; TFRecords are the same images, just a faster container.
    return tr_tbl.lookup(img_id) >= 0


@tf.function
def _keep_val(img_id, img, label):
    return va_tbl.lookup(img_id) >= 0


@tf.function
def _attach_train_label(img_id, img, _label_from_tfrec):
    # Ensure labels exactly match CSV (even if TFRecord target matches, this guarantees equivalence).
    label = tr_tbl.lookup(img_id)
    i = tf.strings.to_hash_bucket_fast(img_id, 2**31 - 1)  # stable per image_id
    return img, label, tf.cast(i, tf.int64)


@tf.function
def _attach_val_label(img_id, img, _label_from_tfrec):
    label = va_tbl.lookup(img_id)
    i = tf.strings.to_hash_bucket_fast(img_id, 2**31 - 1)
    return img, label, tf.cast(i, tf.int64)


def _base_tfrecord_ds(files, parse_fn, cycle_length=8):
    files_ds = tf.data.Dataset.from_tensor_slices(files)

    opts = tf.data.Options()
    opts.experimental_deterministic = False
    opts.experimental_optimization.apply_default_optimizations = True
    opts.experimental_optimization.map_parallelization = True
    opts.experimental_optimization.parallel_batch = True
    opts.experimental_slack = True
    ds = files_ds.with_options(opts)

    ds = ds.interleave(
        lambda f: tf.data.TFRecordDataset(f, num_parallel_reads=tf.data.AUTOTUNE),
        cycle_length=cycle_length,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=False,
    )
    ds = ds.map(parse_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds


def make_ds_from_tfrecords(training: bool, cache_path: str, repeat: bool = False):
    ds = _base_tfrecord_ds(
        TRAIN_TFRECS, _parse_train_example, cycle_length=min(8, len(TRAIN_TFRECS))
    )

    if training:
        ds = ds.filter(_keep_train)
        ds = ds.map(_attach_train_label, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.filter(_keep_val)
        ds = ds.map(_attach_val_label, num_parallel_calls=tf.data.AUTOTUNE)

    # Cache base decoded+resized images + integer labels + stable per-image index once.
    if cache_path:
        ds = ds.cache(cache_path)
    else:
        ds = ds.cache()

    if repeat:
        ds = ds.repeat()

    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)

        @tf.function
        def _aug_and_onehot(img, label, i):
            seed = tf.stack([_SEED_T, i], axis=0)
            img = tf.image.stateless_random_flip_left_right(img, seed=seed)
            y = tf.one_hot(label, depth=_NUM_CLASSES_T)
            return img, y

        ds = ds.map(_aug_and_onehot, num_parallel_calls=tf.data.AUTOTUNE)
    else:

        @tf.function
        def _onehot(img, label, i):
            y = tf.one_hot(label, depth=_NUM_CLASSES_T)
            return img, y

        ds = ds.map(_onehot, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_ds_from_tfrecords(
    training=True, cache_path="/kaggle/working/cache_train_decoded_tfrec", repeat=True
)
# --- Speed fix (preserves semantics): validation does not need .repeat(); validation_steps already bounds it.
val_ds = make_ds_from_tfrecords(
    training=False, cache_path="/kaggle/working/cache_val_decoded_tfrec", repeat=False
)

inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
base = efficientnet_v2.EfficientNetV2B0(
    include_top=False, weights="imagenet", input_tensor=inputs
)
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
    steps_per_execution=32,
)

steps_per_epoch = int(np.ceil(len(tr_df) / BATCH_SIZE))
val_steps = int(np.ceil(len(va_df) / BATCH_SIZE))

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    verbose=1,
)




2026-05-12 01:11:46.315473: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3


2026-05-12 01:11:48.860803: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_10}}
2026-05-12 01:12:21.407280: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144
2026-05-12 01:12:21.408369: W tensorflow/core/framework/op_kernel.cc:1841] OP_REQUIRES failed at example_parsing_ops.cc:98 : INVALID_ARGUMENT:

InvalidArgumentError: Graph execution error:

Detected at node ParseSingleExample/ParseExample/ParseExampleV2 defined at (most recent call last):
<stack traces unavailable>
Error in user-defined function passed to ParallelMapDatasetV2:3 transformation with iterator: Iterator::Root::Prefetch::BatchV2::Map::Shuffle::ForeverRepeat[0]::FileCacheImpl::Map::Filter::Map: Feature: image_id (data type: string) is required but could not be found.
	 [[{{node ParseSingleExample/ParseExample/ParseExampleV2}}]]
	 [[IteratorGetNextAsOptional]] [Op:__inference_multi_step_on_iterator_61908]

In [5]:
# --- Speed fix (preserves core logic): read test set from TFRecords for faster ingestion.
test_ids = sample_df["image_id"].astype(str).to_numpy()
test_id_set = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(test_ids.tolist(), dtype=tf.string),
        values=tf.ones([len(test_ids)], dtype=tf.int32),
    ),
    default_value=tf.constant(0, tf.int32),
)


@tf.function
def _keep_test(img_id, img):
    return test_id_set.lookup(img_id) > 0


def make_test_ds_from_tfrecords():
    ds = _base_tfrecord_ds(
        TEST_TFRECS, _parse_test_example, cycle_length=min(4, len(TEST_TFRECS))
    )
    ds = ds.filter(_keep_test)
    # Drop ids for prediction (submission uses sample_df order, same as original code).
    ds = ds.map(lambda img_id, img: img, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds


test_ds = make_test_ds_from_tfrecords()

pred = model.predict(test_ds, verbose=1)
labels = np.argmax(pred, axis=1).astype(int)

submission = pd.DataFrame({"image_id": test_ids, "label": labels})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)
assert submission.shape[0] == sample_df.shape[0]
assert list(submission.columns) == ["image_id", "label"]
assert submission["image_id"].iloc[0] == sample_df["image_id"].iloc[0]

2026-05-12 01:12:21.715656: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_10}}
2026-05-12 01:12:21.761473: W tensorflow/core/framework/op_kernel.cc:1841] OP_REQUIRES failed at example_parsing_ops.cc:98 : INVALID_ARGUMENT: Feature: image_id (data type: string) is required but could not be found.
2026-05-12 01:12:21.761668: I tensorflow/core/framework/local_rendezvous.cc:4

InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_1_device_/job:localhost/replica:0/task:0/device:CPU:0}} Error in user-defined function passed to ParallelMapDatasetV2:31 transformation with iterator: Iterator::Root::Prefetch::BatchV2::Map::Filter::Map: Feature: image_id (data type: string) is required but could not be found.
	 [[{{function_node __inference__parse_test_example_61961}}{{node ParseSingleExample/ParseExample/ParseExampleV2}}]] [Op:IteratorGetNext] name: 